In [2]:
library(spacexr)
library(Matrix)

# single-cell analysis package
library(Seurat)

# plotting and data science packages
library(repr)
library(readxl)
library(stringr)
library(dplyr) 
library(tidyverse)
library(ggplot2)
library(cowplot)
library(patchwork)

## Single-cell data

In [3]:
outDir <- file.path("./../data/")
scRNAseq_Ob <- readRDS(file.path(outDir, "integrated_scRNASeq_data.rds"))
scRNAseq_Ob

An object of class Seurat 
19915 features across 16682 samples within 1 assay 
Active assay: RNA (19915 features, 3000 variable features)
 3 layers present: data, counts, scale.data
 4 dimensional reductions calculated: pca, tsne, umap, integrated.cca

In [4]:
scRNAseq_Ob@meta.data$cell_type <- scRNAseq_Ob@meta.data$cell_type_annotation
scRNAseq_Ob@meta.data <- scRNAseq_Ob@meta.data %>% 
                            mutate(cell_type = recode(cell_type, 'Fibro1 (EIF4A3, STAR)' = 'Fibroblasts', 
                                                      'Fibro2 (RBP1, DCN)' = 'Fibroblasts',
                                                      'Fibro3 (RAMP1, CFD)' = 'Fibroblasts',
                                                      "Fibro4 (CCL2)" = "Fibroblasts",
                                                      'Fibro5 (FN1, COL3A1)' = 'Fibroblasts',
                                                      'B/Plasma cells' = 'B_Plasma_cells',
                                                      'Endothelial cells' = 'Endothelial_cells',
                                                      'Mesothelial cells' = 'Mesothelial_cells',
                                                      'T cells' = 'T_cells',
                                                      'Tumour cells' = 'Tumour_cells'
                                                      ))

In [5]:
cell_types <- scRNAseq_Ob@meta.data$cell_type
names(cell_types) <- rownames(scRNAseq_Ob@meta.data) # create cell_types named list
cell_types <- as.factor(cell_types) # convert to factor data type

nUMI <- scRNAseq_Ob@meta.data$nCount_RNA 
names(nUMI) <- rownames(scRNAseq_Ob@meta.data) # create nUMI named list

In [6]:
### Create the Reference object
reference <- Reference(scRNAseq_Ob[["RNA"]]$counts, cell_types, nUMI, n_max_cells = 11000)
## Examine reference object (optional)
print(dim(reference@counts))
table(reference@cell_types) #number of occurences for each cell type

[1] 19915 16682



   B_Plasma_cells Endothelial_cells       Fibroblasts       Macrophages 
               28              1639             10819               314 
Mesothelial_cells    Myofibroblasts           T_cells      Tumour_cells 
              249              2235               787               611 

## ST Data

In [7]:
stOb <- readRDS(file.path("./../data", "unprocessed_STdata.rds"))
stOb

An object of class Seurat 
33538 features across 20123 samples within 1 assay 
Active assay: Spatial (33538 features, 0 variable features)
 8 layers present: counts.1, counts.2, counts.3, counts.4, counts.5, counts.6, counts.7, counts.8
 8 spatial fields of view present: P1 P2 P3 P4 P5 P6 P7 P8

In [8]:
puck_lst <- list()
for (i in seq(1:8)){
    counts <- GetAssayData(object = stOb, layer = Layers(stOb[["Spatial"]])[i])
    
    coords <-  GetTissueCoordinates(stOb@images[[paste0("P", i)]])
    coords$cell <- NULL

    nUMI <- filter(stOb@meta.data, patient == paste0("P", i))$nCount_Spatial
    names(nUMI) <- rownames(filter(stOb@meta.data, patient == paste0("P", i)))

    ### Create SpatialRNA object
    puck <- SpatialRNA(coords, counts, nUMI)
    
    ## Examine SpatialRNA object (optional)
    print(dim(puck@counts)) 

    puck_lst <- append(puck_lst, puck)
    
}

[1] 33538  2940
[1] 33538  1935
[1] 33538  1504
[1] 33538  3007
[1] 33538  2124
[1] 33538  1801
[1] 33538  3632
[1] 33538  3180


## RCTD

In [9]:
## Patient 1
RCTD_p1 <- create.RCTD(puck_lst[[1]], reference, max_cores = 2)
RCTD_p1 <- run.RCTD(RCTD_p1, doublet_mode = 'full')
results_p1 <- RCTD_p1@results
# normalize the cell type proportions to sum to 1.
norm_weights_p1 = normalize_weights(results_p1$weights) 
write.csv(norm_weights_p1, "./results/GSM6506110_SP1.csv")

Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 16682

process_cell_type_info: number of genes in reference: 19915




   B_Plasma_cells Endothelial_cells       Fibroblasts       Macrophages 
               28              1639             10819               314 
Mesothelial_cells    Myofibroblasts           T_cells      Tumour_cells 
              249              2235               787               611 


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.6 GiB”
End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 42

get_de_genes: Endothelial_cells found DE genes: 118

get_de_genes: Fibroblasts found DE genes: 83

get_de_genes: Macrophages found DE genes: 188

get_de_genes: Mesothelial_cells found DE genes: 68

get_de_genes: Myofibroblasts found DE genes: 135

get_de_genes: T_cells found DE genes: 148

get_de_genes: Tumour_cells found DE genes: 99

get_de_genes: total DE genes: 772

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 80

get_de_genes: Endothelial_cells found DE genes: 281

get_de_genes: Fibroblasts found DE genes: 214

get_de_genes: Macrophages found DE genes: 344

get_de_genes: Mesothelial_cells found DE genes: 179

get_de_genes: Myofibroblasts found DE genes: 267

ge

In [10]:
## Patient 2
RCTD_p2 <- create.RCTD(puck_lst[[2]], reference, max_cores = 2)
RCTD_p2 <- run.RCTD(RCTD_p2, doublet_mode = 'full')
results_p2 <- RCTD_p2@results
# normalize the cell type proportions to sum to 1.
norm_weights_p2 = normalize_weights(results_p2$weights) 
write.csv(norm_weights_p2, "./results/GSM6506111_SP2.csv")

Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 16682

process_cell_type_info: number of genes in reference: 19915




   B_Plasma_cells Endothelial_cells       Fibroblasts       Macrophages 
               28              1639             10819               314 
Mesothelial_cells    Myofibroblasts           T_cells      Tumour_cells 
              249              2235               787               611 


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.6 GiB”
End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 42

get_de_genes: Endothelial_cells found DE genes: 118

get_de_genes: Fibroblasts found DE genes: 82

get_de_genes: Macrophages found DE genes: 190

get_de_genes: Mesothelial_cells found DE genes: 67

get_de_genes: Myofibroblasts found DE genes: 134

get_de_genes: T_cells found DE genes: 147

get_de_genes: Tumour_cells found DE genes: 100

get_de_genes: total DE genes: 771

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 80

get_de_genes: Endothelial_cells found DE genes: 281

get_de_genes: Fibroblasts found DE genes: 213

get_de_genes: Macrophages found DE genes: 347

get_de_genes: Mesothelial_cells found DE genes: 178

get_de_genes: Myofibroblasts found DE genes: 266

g

In [11]:
## Patient 3
RCTD_p3 <- create.RCTD(puck_lst[[3]], reference, max_cores = 2)
RCTD_p3 <- run.RCTD(RCTD_p3, doublet_mode = 'full')
results_p3 <- RCTD_p3@results
# normalize the cell type proportions to sum to 1.
norm_weights_p3 = normalize_weights(results_p3$weights) 
write.csv(norm_weights_p3, "./results/GSM6506112_SP3.csv")

Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 16682

process_cell_type_info: number of genes in reference: 19915




   B_Plasma_cells Endothelial_cells       Fibroblasts       Macrophages 
               28              1639             10819               314 
Mesothelial_cells    Myofibroblasts           T_cells      Tumour_cells 
              249              2235               787               611 


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.6 GiB”
End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 42

get_de_genes: Endothelial_cells found DE genes: 118

get_de_genes: Fibroblasts found DE genes: 83

get_de_genes: Macrophages found DE genes: 189

get_de_genes: Mesothelial_cells found DE genes: 69

get_de_genes: Myofibroblasts found DE genes: 135

get_de_genes: T_cells found DE genes: 147

get_de_genes: Tumour_cells found DE genes: 99

get_de_genes: total DE genes: 773

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 79

get_de_genes: Endothelial_cells found DE genes: 281

get_de_genes: Fibroblasts found DE genes: 214

get_de_genes: Macrophages found DE genes: 346

get_de_genes: Mesothelial_cells found DE genes: 180

get_de_genes: Myofibroblasts found DE genes: 267

ge

In [12]:
## Patient 4
RCTD_p4 <- create.RCTD(puck_lst[[4]], reference, max_cores = 2)
RCTD_p4 <- run.RCTD(RCTD_p4, doublet_mode = 'full')
results_p4 <- RCTD_p4@results
# normalize the cell type proportions to sum to 1.
norm_weights_p4 = normalize_weights(results_p4$weights) 
write.csv(norm_weights_p4, "./results/GSM6506113_SP4.csv")

Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 16682

process_cell_type_info: number of genes in reference: 19915




   B_Plasma_cells Endothelial_cells       Fibroblasts       Macrophages 
               28              1639             10819               314 
Mesothelial_cells    Myofibroblasts           T_cells      Tumour_cells 
              249              2235               787               611 


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.6 GiB”
End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 40

get_de_genes: Endothelial_cells found DE genes: 117

get_de_genes: Fibroblasts found DE genes: 83

get_de_genes: Macrophages found DE genes: 189

get_de_genes: Mesothelial_cells found DE genes: 68

get_de_genes: Myofibroblasts found DE genes: 135

get_de_genes: T_cells found DE genes: 143

get_de_genes: Tumour_cells found DE genes: 100

get_de_genes: total DE genes: 766

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 77

get_de_genes: Endothelial_cells found DE genes: 280

get_de_genes: Fibroblasts found DE genes: 214

get_de_genes: Macrophages found DE genes: 346

get_de_genes: Mesothelial_cells found DE genes: 179

get_de_genes: Myofibroblasts found DE genes: 267

g

In [13]:
## Patient 5
RCTD_p5 <- create.RCTD(puck_lst[[5]], reference, max_cores = 2)
RCTD_p5 <- run.RCTD(RCTD_p5, doublet_mode = 'full')
results_p5 <- RCTD_p5@results
# normalize the cell type proportions to sum to 1.
norm_weights_p5 = normalize_weights(results_p5$weights) 
write.csv(norm_weights_p5, "./results/GSM6506114_SP5.csv")

Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 16682

process_cell_type_info: number of genes in reference: 19915




   B_Plasma_cells Endothelial_cells       Fibroblasts       Macrophages 
               28              1639             10819               314 
Mesothelial_cells    Myofibroblasts           T_cells      Tumour_cells 
              249              2235               787               611 


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.6 GiB”
End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 42

get_de_genes: Endothelial_cells found DE genes: 118

get_de_genes: Fibroblasts found DE genes: 82

get_de_genes: Macrophages found DE genes: 190

get_de_genes: Mesothelial_cells found DE genes: 67

get_de_genes: Myofibroblasts found DE genes: 134

get_de_genes: T_cells found DE genes: 147

get_de_genes: Tumour_cells found DE genes: 100

get_de_genes: total DE genes: 771

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 80

get_de_genes: Endothelial_cells found DE genes: 281

get_de_genes: Fibroblasts found DE genes: 213

get_de_genes: Macrophages found DE genes: 347

get_de_genes: Mesothelial_cells found DE genes: 178

get_de_genes: Myofibroblasts found DE genes: 266

g

In [14]:
## Patient 6
RCTD_p6 <- create.RCTD(puck_lst[[6]], reference, max_cores = 2)
RCTD_p6 <- run.RCTD(RCTD_p6, doublet_mode = 'full')
results_p6 <- RCTD_p6@results
# normalize the cell type proportions to sum to 1.
norm_weights_p6 = normalize_weights(results_p6$weights) 
write.csv(norm_weights_p6, "./results/GSM6506115_SP6.csv")

Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 16682

process_cell_type_info: number of genes in reference: 19915




   B_Plasma_cells Endothelial_cells       Fibroblasts       Macrophages 
               28              1639             10819               314 
Mesothelial_cells    Myofibroblasts           T_cells      Tumour_cells 
              249              2235               787               611 


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.6 GiB”
End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 42

get_de_genes: Endothelial_cells found DE genes: 118

get_de_genes: Fibroblasts found DE genes: 83

get_de_genes: Macrophages found DE genes: 188

get_de_genes: Mesothelial_cells found DE genes: 68

get_de_genes: Myofibroblasts found DE genes: 135

get_de_genes: T_cells found DE genes: 147

get_de_genes: Tumour_cells found DE genes: 99

get_de_genes: total DE genes: 771

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 79

get_de_genes: Endothelial_cells found DE genes: 281

get_de_genes: Fibroblasts found DE genes: 214

get_de_genes: Macrophages found DE genes: 345

get_de_genes: Mesothelial_cells found DE genes: 179

get_de_genes: Myofibroblasts found DE genes: 267

ge

In [15]:
## Patient 7
RCTD_p7 <- create.RCTD(puck_lst[[7]], reference, max_cores = 2)
RCTD_p7 <- run.RCTD(RCTD_p7, doublet_mode = 'full')
results_p7 <- RCTD_p7@results
# normalize the cell type proportions to sum to 1.
norm_weights_p7 = normalize_weights(results_p7$weights) 
write.csv(norm_weights_p7, "./results/GSM6506116_SP7.csv")

Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 16682

process_cell_type_info: number of genes in reference: 19915




   B_Plasma_cells Endothelial_cells       Fibroblasts       Macrophages 
               28              1639             10819               314 
Mesothelial_cells    Myofibroblasts           T_cells      Tumour_cells 
              249              2235               787               611 


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.6 GiB”
End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 42

get_de_genes: Endothelial_cells found DE genes: 118

get_de_genes: Fibroblasts found DE genes: 83

get_de_genes: Macrophages found DE genes: 190

get_de_genes: Mesothelial_cells found DE genes: 67

get_de_genes: Myofibroblasts found DE genes: 135

get_de_genes: T_cells found DE genes: 148

get_de_genes: Tumour_cells found DE genes: 99

get_de_genes: total DE genes: 773

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 80

get_de_genes: Endothelial_cells found DE genes: 281

get_de_genes: Fibroblasts found DE genes: 214

get_de_genes: Macrophages found DE genes: 347

get_de_genes: Mesothelial_cells found DE genes: 178

get_de_genes: Myofibroblasts found DE genes: 267

ge

In [16]:
## Patient 8
RCTD_p8 <- create.RCTD(puck_lst[[8]], reference, max_cores = 2)
RCTD_p8 <- run.RCTD(RCTD_p8, doublet_mode = 'full')
results_p8 <- RCTD_p8@results
# normalize the cell type proportions to sum to 1.
norm_weights_p8 = normalize_weights(results_p8$weights) 
write.csv(norm_weights_p8, "./results/GSM6506117_SP8.csv")

Begin: process_cell_type_info

process_cell_type_info: number of cells in reference: 16682

process_cell_type_info: number of genes in reference: 19915




   B_Plasma_cells Endothelial_cells       Fibroblasts       Macrophages 
               28              1639             10819               314 
Mesothelial_cells    Myofibroblasts           T_cells      Tumour_cells 
              249              2235               787               611 


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.6 GiB”
End: process_cell_type_info

create.RCTD: getting regression differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 42

get_de_genes: Endothelial_cells found DE genes: 118

get_de_genes: Fibroblasts found DE genes: 83

get_de_genes: Macrophages found DE genes: 190

get_de_genes: Mesothelial_cells found DE genes: 69

get_de_genes: Myofibroblasts found DE genes: 135

get_de_genes: T_cells found DE genes: 148

get_de_genes: Tumour_cells found DE genes: 99

get_de_genes: total DE genes: 775

create.RCTD: getting platform effect normalization differentially expressed genes: 

get_de_genes: B_Plasma_cells found DE genes: 80

get_de_genes: Endothelial_cells found DE genes: 281

get_de_genes: Fibroblasts found DE genes: 214

get_de_genes: Macrophages found DE genes: 347

get_de_genes: Mesothelial_cells found DE genes: 180

get_de_genes: Myofibroblasts found DE genes: 267

ge